# Huấn luyện 3D Gaussian Splatting (Train from scratch)

Notebook này thực hiện huấn luyện các scene của tập `private_set1` từ bộ dữ liệu đã được tiền xử lý (converted).

In [ ]:
# 1. Clone và cài đặt 3DGS
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd gaussian-splatting

# Thay thế simple-knn bằng bản fix lỗi build của camenduru
!rm -rf submodules/simple-knn
!git clone https://github.com/camenduru/simple-knn.git submodules/simple-knn

# Patch để tránh lỗi Read-only file system khi chuyển đổi points3D.bin sang points3D.ply
import os
readers_path = 'scene/dataset_readers.py'
if os.path.exists(readers_path):
    with open(readers_path, 'r') as f:
        code = f.read()
    old_code = 'ply_path = os.path.join(path, "sparse/0/points3D.ply")'
    new_code = '''ply_path = os.path.join(path, "sparse/0/points3D.ply")
    if not os.path.exists(ply_path):
        import hashlib
        path_hash = hashlib.md5(path.encode()).hexdigest()
        ply_path_tmp = os.path.join("/tmp", f"points3D_{path_hash}.ply")
        if os.path.exists(ply_path_tmp):
            ply_path = ply_path_tmp
        else:
            ply_path = ply_path_tmp'''
    if old_code in code:
        code = code.replace(old_code, new_code)
        with open(readers_path, 'w') as f:
            f.write(code)
        print('Patched dataset_readers.py successfully!')

!pip install -q plyfile tqdm opencv-python joblib pillow lpips
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn

In [ ]:
import os
import subprocess

# 2. Cấu hình đường dẫn và scene cần train
DATASET_ROOT = '/kaggle/input/converted-bts-dataset/private_set1'
OUTPUT_ROOT = '/kaggle/working/outputs'

# Danh sách scene cần train. Để None hoặc [] nếu muốn train TẤT CẢ các scene trong folder
TARGET_SCENES = ['HCM0204']  # Ví dụ: ['HCM0204', 'HCM0181']

# Các tham số training
ITERATIONS = 15000

if not os.path.exists(DATASET_ROOT):
    print(f"Thư mục {DATASET_ROOT} không tồn tại. Vui lòng kiểm tra lại data input!")
else:
    all_scenes = sorted(os.listdir(DATASET_ROOT))
    
    # Lọc scene theo TARGET_SCENES
    if TARGET_SCENES:
        scenes = [s for s in all_scenes if s in TARGET_SCENES]
    else:
        scenes = all_scenes
        
    print(f"Tìm thấy {len(all_scenes)} scenes trong dataset.")
    print(f"Sẽ thực hiện train {len(scenes)} scenes: {scenes}")
    
    for scene in scenes:
        print("\n" + "="*60)
        print(f"TRAINING SCENE: {scene}")
        print("="*60)
        
        scene_path = os.path.join(DATASET_ROOT, scene, 'train')
        output_dir = os.path.join(OUTPUT_ROOT, scene)
        
        command = [
            "python", "train.py",
            "-s", scene_path,
            "-m", output_dir,
            "--eval",
            "--iterations", str(ITERATIONS),
            "--test_iterations", "7000", str(ITERATIONS),
            "--save_iterations", str(ITERATIONS),
            "--checkpoint_iterations", str(ITERATIONS),
            "--position_lr_max_steps", str(ITERATIONS),
            "--disable_viewer"
        ]
        
        # Chạy lệnh train và in log trực tiếp
        subprocess.run(command, check=True)
        print(f"Hoàn thành train 3DGS cho scene {scene}!")